# Trabalho Grau B - Reconhecimento de imagem e transfer learning

## Integrantes

- Arthur Schallenberger
- Giovani de Souza
- Leonardo Fronza
- Renan Milech Pereira

---

**Docente:** Prof. Gabriel de Oliveira Ramos

# 2 Modelagem
  
## 2.1 Descrição do Problema e Dataset

## 2.2 Análise das Classes e Dados


## 2.3 Pré-processamento


## 2.4 Arquitetura das Redes Neurais

Para o trabalho, foram consideradas duas abordagens complementares: uma CNN construída do zero, usada como linha de base, e redes pré-treinadas com *transfer learning*, especialmente EfficientNetB0 e MobileNetV2 (que são redes leves e eficientes e que uma delas será escolhida para o transfer learning).

### 2.4.1 CNN própria

A CNN própria foi pensada para ser simples, estável e fácil de interpretar. A arquitetura proposta segue a lógica de extração progressiva de características:

- **Camada de entrada:** imagens redimensionadas para um formato fixo, como 224 x 224 x 3.
- **Blocos convolucionais:** 3 blocos com convoluções 2D, ativação ReLU e *padding* igual.
- **Pooling:** *MaxPooling2D* após cada bloco convolucional para reduzir dimensionalidade e manter as informações mais relevantes.
- **Regularização:** *Dropout* entre os blocos e antes da saída para reduzir *overfitting*.
- **Classificação final:** camadas densas com *softmax* na saída, uma neurônio por classe.

Uma configuração coerente para essa CNN é:

- Bloco 1: 32 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 2: 64 filtros, convolução 3 x 3, ReLU, MaxPooling
- Bloco 3: 128 filtros, convolução 3 x 3, ReLU, MaxPooling
- *Flatten* ou *GlobalAveragePooling2D*
- *Dense* final com *softmax*

Essa estrutura é suficiente para capturar padrões visuais básicos e serve como referência para comparar com as redes pré-treinadas.

### 2.4.2 Transfer learning

A rede de *transfer learning* escolhida para o experimento principal foi a **EfficientNetB0**, por apresentar bom equilíbrio entre desempenho e custo computacional. A **MobileNetV2** foi mantida como comparação leve, útil quando a prioridade é reduzir parâmetros e acelerar inferência.

A estratégia adotada para a EfficientNetB0 (experimento principal):

1. Carregar os pesos pré-treinados no ImageNet.
2. Congelar a base convolucional nas primeiras etapas.
3. Adicionar uma cabeça de classificação específica para as classes do conjunto de dados.
4. Se necessário, liberar parte das últimas camadas para *fine-tuning*.

### 2.4.3 Justificativa das escolhas

As escolhas arquiteturais foram feitas considerando o tamanho do conjunto de dados e o objetivo de classificação de imagens esportivas:

- A **CNN própria** funciona como baseline e permite avaliar o quanto o problema pode ser resolvido sem conhecimento prévio transferido.
- **EfficientNetB0** tende a oferecer melhor relação entre profundidade, eficiência e generalização, sendo uma boa candidata para maior acurácia.
- **MobileNetV2** é uma alternativa mais leve, com menor custo de processamento, útil para comparação e para cenários com limitação de recursos.
- O uso de **Dropout** e de *MaxPooling* ajuda a controlar o sobreajuste e a reduzir o tamanho das representações intermediárias.
- A ativação **ReLU** é adequada por ser simples, eficiente e amplamente usada em CNNs modernas.

### 2.4.4 Comparação das arquiteturas

| Arquitetura | Número de camadas | Convoluções | Pooling | Dropout | Função de ativação | Vantagem principal |
| --- | --- | --- | --- | --- | --- | --- |
| CNN própria | 3 blocos convolucionais + classificadores | 3 x 3 | MaxPooling2D | Sim | ReLU / Softmax | Baseline simples e interpretável |
| EfficientNetB0 | Backbone pré-treinado + cabeça densa | Convoluções otimizadas pela família EfficientNet | GlobalAveragePooling2D ou pooling implícito | Sim | Swish/ReLU + Softmax | Melhor equilíbrio entre desempenho e custo |
| MobileNetV2 | Backbone pré-treinado + cabeça densa | Convoluções separáveis | GlobalAveragePooling2D | Sim | ReLU6 + Softmax | Menor custo computacional |

### 2.4.5 Diagrama simplificado

```mermaid
flowchart LR
    A[Imagem de entrada\n224 x 224 x 3] --> B{Estratégia}
    B --> C[CNN própria\nConv 3x3 + ReLU\nMaxPooling + Dropout]
    B --> D[EfficientNetB0\nbase congelada + cabeça densa]
    B --> E[MobileNetV2\nbase congelada + cabeça leve]
    C --> F[Softmax\nclassificação]
    D --> F
    E --> F
```

Em resumo, a CNN própria fornece a linha de base do estudo, enquanto EfficientNetB0 e MobileNetV2 representam as melhores alternativas de *transfer learning* para comparar desempenho, robustez e custo de execução.